# runtime

> Execute code and edit values in the live kernel namespace

The *owner's* write path (agents get their own, restricted one in `agent`): `run` executes
through IPython's `run_cell` — `scope='isolated'` runs on a scratch copy instead, showing
the result without touching state — and `set_value` assigns to any accessor-addressed
lvalue. Every global run is appended to a session notebook under `paar_sessions/`, so the
exec-bar history is replayable `.ipynb`, listed and reloaded by the Sessions panel.
`complete` serves tab-completion from the real kernel (or a namespace-walking fallback when
headless).

In [ ]:
#| default_exp runtime

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import tempfile
from pathlib import Path
import paar.runtime as R
from paar.runtime import run, ExecResult, set_value, complete, log_run, list_sessions, read_session, current_session
R.SESSION_DIR = Path(tempfile.mkdtemp())   # keep example transcripts out of nbs/

class _Shell:
    "Minimal IPython stand-in whose run_cell execs into a shared user_ns."
    def __init__(self): self.user_ns = {}
    def run_cell(self, code, store_history=True):
        import types
        ns = self.user_ns
        err = None; result = None
        try:
            block = compile(code, '<t>', 'exec')
            exec(block, ns)
            # emulate IPython binding `_` for a trailing expression
            import ast
            body = ast.parse(code).body
            if body and isinstance(body[-1], ast.Expr):
                result = eval(compile(ast.Expression(body[-1].value), '<t>', 'eval'), ns)
                ns['_'] = result
        except Exception as e:
            err = e
        return types.SimpleNamespace(result=result, error_in_exec=err,
                                     error_before_exec=None, success=err is None)
    def complete(self, text=None, line=None, cursor_pos=None):
        import re
        q = text if text is not None else (line or '')[:cursor_pos]
        tok = re.split(r'[^\w.]', q)[-1]
        return tok, [k for k in self.user_ns if k.startswith(tok)]
R.SESSION_DIR = Path(tempfile.mkdtemp())

def test_run_global_assignment_mutates_ns():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('x = 5')
    assert isinstance(r, ExecResult) and r.ok is True
    assert r.result is None and sh.user_ns['x'] == 5   # assignment: no result row
def test_run_global_expression_makes_result_row():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('3 + 4')
    assert r.ok is True and r.result is not None
    assert r.result.name == 'result' and r.result.value == '7' and r.result.accessor == ('_',)
def test_run_global_modify_existing():
    sh = _Shell(); sh.user_ns['n'] = 1; R.get_ipython = lambda: sh
    run('n = n + 41'); assert sh.user_ns['n'] == 42
def test_run_error_is_captured():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('1/0')
    assert r.ok is False and 'ZeroDivisionError' in r.error
def test_run_no_ipython():
    R.get_ipython = lambda: None
    r = run('x=1'); assert r.ok is False and 'no IPython' in r.error
def test_run_isolated_no_leak():
    sh = _Shell(); sh.user_ns['seed'] = 10; R.get_ipython = lambda: sh
    r = run('y = seed + 1\ny', scope='isolated')
    assert r.ok is True and r.result is not None and r.result.value == '11'
    assert 'y' not in sh.user_ns                       # isolated: no leak into globals
    assert r.result.is_container is False and r.result.accessor == ()   # flat, non-expandable
def test_run_isolated_statement_only():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('z = 1', scope='isolated')
    assert r.ok is True and r.result is None and 'z' not in sh.user_ns
def test_run_isolated_error():
    sh = _Shell(); R.get_ipython = lambda: sh
    r = run('1/0', scope='isolated')
    assert r.ok is False and 'ZeroDivisionError' in r.error and sh.user_ns == {}   # error captured, no leak

def test_set_value_toplevel():
    sh = _Shell(); sh.user_ns['x'] = 1; R.get_ipython = lambda: sh
    assert set_value(('x',), '99') is None and sh.user_ns['x'] == 99
def test_set_value_dict_item():
    sh = _Shell(); sh.user_ns['d'] = {'k': 1}; R.get_ipython = lambda: sh
    assert set_value(('d', 0), '2') is None and sh.user_ns['d']['k'] == 2   # accessor ('d',0)->d['k']
def test_set_value_list_element():
    sh = _Shell(); sh.user_ns['L'] = [10, 20]; R.get_ipython = lambda: sh
    assert set_value(('L', 1), '21') is None and sh.user_ns['L'][1] == 21
def test_set_value_bad_target_errors():
    sh = _Shell(); sh.user_ns['s'] = {1, 2}; R.get_ipython = lambda: sh
    err = set_value(('s', 0), '9')          # set element path is display-only, not an lvalue
    assert err is not None and 's' in sh.user_ns   # error string, no crash

def test_complete_names():
    sh=_Shell(); sh.user_ns.update({'alpha':1,'alto':2,'beta':3}); R.get_ipython=lambda: sh
    r = complete('al', 2)
    assert r['from']==0 and set(r['matches'])=={'alpha','alto'}   # from = token start offset
def test_complete_no_ipython():
    R.get_ipython=lambda: None                       # no kernel -> namespace completion fallback
    import paar.bridge as BR; BR.set_namespace({'xylophone': 1})
    try:
        assert 'xylophone' in complete('xy', 2)['matches']   # completes names from the inspected ns
        assert complete('', 0) == {'from':0, 'matches':[]}   # empty token -> nothing
    finally:
        BR.set_namespace(None)
def test_current_session():
    from paar.runtime import current_session
    assert current_session().startswith('session_')
def test_run_logs_session_notebook():
    from fastcore.nbio import read_nb
    R.SESSION_DIR = Path(tempfile.mkdtemp())          # isolate this test
    sh=_Shell(); R.get_ipython=lambda: sh
    run('a = 1'); run('b = 2')
    files = list(R.SESSION_DIR.glob('session_*.ipynb'))
    assert len(files)==1                               # one notebook for the session
    nb = read_nb(files[0]); srcs=[c['source'] for c in nb['cells']]
    assert 'a = 1' in srcs and 'b = 2' in srcs         # each run appended as a cell
def test_isolated_run_not_logged():
    R.SESSION_DIR = Path(tempfile.mkdtemp())
    sh=_Shell(); R.get_ipython=lambda: sh
    run('c = 3', scope='isolated')
    assert list(R.SESSION_DIR.glob('session_*.ipynb')) == []   # isolated runs are not persisted

def test_list_and_read_sessions():
    R.SESSION_DIR = Path(tempfile.mkdtemp())
    sh=_Shell(); R.get_ipython=lambda: sh
    run('m = 1'); run('m = 2')
    ss = list_sessions()
    assert len(ss)==1 and ss[0][1]==2                    # one session notebook, 2 code cells
    cells = read_session(ss[0][0])
    assert cells == ['m = 1', 'm = 2']                   # code sources, in order
def test_read_session_rejects_bad_name():
    assert read_session('../secret')==[] and read_session('nope')==[]   # traversal + missing -> []

for t in [test_run_global_assignment_mutates_ns, test_run_global_expression_makes_result_row,
          test_run_global_modify_existing, test_run_error_is_captured, test_run_no_ipython,
          test_run_isolated_no_leak, test_run_isolated_statement_only, test_run_isolated_error,
          test_set_value_toplevel, test_set_value_dict_item, test_set_value_list_element,
          test_set_value_bad_target_errors,
          test_complete_names, test_complete_no_ipython, test_current_session,
          test_run_logs_session_notebook, test_isolated_run_not_logged,
          test_list_and_read_sessions, test_read_session_rejects_bad_name]: t()

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass
import ast, datetime, re
from pathlib import Path
from paar.core import VarInfo
from paar.snapshot import _var_info, _walk
from paar.providers import value_str
from IPython.utils.capture import capture_output
from fastcore.nbio import read_nb, write_nb, new_nb, mk_cell
try: from IPython import get_ipython
except Exception:
    def get_ipython(): return None

@dataclass
class ExecResult:
    "Outcome of running code: last value as an inspector row, captured stdout, and error text."
    ok: bool
    result: VarInfo|None = None
    stdout: str = ''
    error: str|None = None

def _fmt_err(res):
    "Formatted exception text from a run_cell result, or None."
    e = getattr(res, 'error_in_exec', None) or getattr(res, 'error_before_exec', None)
    return f'{type(e).__name__}: {e}' if e is not None else None

def _exec_capture(code, ns):
    "Exec statements in ns; return the value of a trailing expression, else None."
    block = ast.parse(code)
    value = None
    if block.body and isinstance(block.body[-1], ast.Expr):
        last = ast.Expression(block.body.pop().value)
        exec(compile(block, '<paar>', 'exec'), ns)
        value = eval(compile(last, '<paar>', 'eval'), ns)
    else:
        exec(compile(block, '<paar>', 'exec'), ns)
    return value

def _flat_result(val):
    "A non-expandable result row (isolated mode leaves nothing reachable in user_ns)."
    return VarInfo(name='result', type=type(val).__name__, value=value_str(val))

def _ns_complete(code, pos):
    "Namespace completion fallback when there is no IPython kernel (headless serve())."
    import builtins, keyword
    from paar.bridge import _ns
    ns = _ns()
    tok = re.search(r'[\w.]*$', code[:pos]).group(0)
    if not tok: return {'from': pos, 'matches': []}
    if '.' in tok:
        base, _, prefix = tok.rpartition('.')
        try: obj = eval(base, {'__builtins__': builtins}, ns)     # evaluated in the inspected namespace only
        except Exception: return {'from': pos, 'matches': []}
        cands = [a for a in dir(obj) if a.startswith(prefix) and (prefix or not a.startswith('_'))]
        return {'from': pos - len(prefix), 'matches': sorted(cands)[:50]}
    names = set(ns) | set(dir(builtins)) | set(keyword.kwlist)
    cands = [n for n in names if n.startswith(tok)]
    return {'from': pos - len(tok), 'matches': sorted(cands)[:50]}

def _ip_complete(ip, code, pos):
    "Real IPython InteractiveShell.complete(text, line, cursor) -> (pre, matches); None if that signature doesn't apply."
    bol = code.rfind('\n', 0, pos) + 1                     # current physical line + cursor within it:
    eol = code.find('\n', pos); eol = len(code) if eol < 0 else eol   # gives IPython the `import ...`
    line, cur = code[bol:eol], pos - bol                   # context so venv modules complete too
    token = re.search(r'[\w.]*$', code[:pos]).group(0)
    cmp = getattr(ip, 'Completer', None)   # real kernel: legacy ip.complete() returns nothing while Jedi is on
    old = getattr(cmp, 'use_jedi', None)
    if cmp is not None: cmp.use_jedi = False
    try: pre, matches = ip.complete(token, line, cur)
    except Exception: return None
    finally:
        if old is not None: cmp.use_jedi = old   # restore so the user's own tab-completion keeps Jedi
    return {'from': pos - len(pre), 'matches': list(matches)[:50]}

def complete(code, pos):
    "Completions for the token ending at offset `pos` -> {'from': int, 'matches': [str]}. Handles execnb's CaptureShell completer (single-arg, what `serve` runs on), a real IPython kernel, and a headless namespace fallback."
    ip = get_ipython()
    if ip is None: return _ns_complete(code, pos)
    prefix = code[:pos]
    try: matches = ip.complete(prefix)   # execnb CaptureShell.complete(c) -> [tail matches]
    except Exception: matches = None
    if isinstance(matches, list):        # tail after the last dot is what those matches replace
        tail = re.search(r'[\w.]*$', prefix).group(0).rpartition('.')[-1]
        return {'from': pos - len(tail), 'matches': matches[:50]}
    return _ip_complete(ip, code, pos) or {'from': pos, 'matches': []}

SESSION_DIR = Path('paar_sessions')
_SESSION = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

def _session_path():
    SESSION_DIR.mkdir(parents=True, exist_ok=True)
    return SESSION_DIR/f'session_{_SESSION}.ipynb'

def current_session(): return f'session_{_SESSION}'

def log_run(code, path=None, cell_type='code'):
    "Append `code` as a cell to a session notebook (default: this owner session's; best-effort, never raises)."
    try:
        p = Path(path) if path else _session_path()
        p.parent.mkdir(parents=True, exist_ok=True)
        nb = read_nb(p) if p.exists() else new_nb([])
        nb['cells'].append(mk_cell(code, cell_type))
        write_nb(nb, p)
    except Exception: pass

def list_sessions():
    "Saved session notebooks (owner `session_*` and agent `agent_*`), newest first, as [(stem, n_code_cells)]."
    if not SESSION_DIR.exists(): return []
    out = []
    for p in sorted([*SESSION_DIR.glob('session_*.ipynb'), *SESSION_DIR.glob('agent_*.ipynb')],
                    key=lambda p: p.name, reverse=True):
        try: n = sum(c.get('cell_type')=='code' for c in read_nb(p)['cells'])
        except Exception: n = 0
        out.append((p.stem, n))
    return out

def read_session(name):
    "Code-cell sources for session `name` (a session_*/agent_* stem); [] if missing or name is unsafe."
    if '/' in name or chr(92) in name or not name.startswith(('session_', 'agent_')): return []
    p = SESSION_DIR/f'{name}.ipynb'
    if not p.exists(): return []
    try: return [c['source'] for c in read_nb(p)['cells'] if c.get('cell_type')=='code']
    except Exception: return []

def run(code, scope='global'):
    "Run `code` in the kernel; scope='global' mutates user_ns, 'isolated' uses a scratch copy."
    ip = get_ipython()
    if ip is None: return ExecResult(ok=False, error='no IPython kernel')
    if scope == 'isolated':
        ns = dict(ip.user_ns)
        try:
            with capture_output() as cap:
                val = _exec_capture(code, ns)
            return ExecResult(ok=True, result=None if val is None else _flat_result(val), stdout=cap.stdout)
        except Exception as e:
            return ExecResult(ok=False, error=f'{type(e).__name__}: {e}')
    log_run(code)
    with capture_output() as cap:
        res = ip.run_cell(code, store_history=True)
    err = _fmt_err(res)
    result = _var_info('result', res.result, ('_',), '_') if res.result is not None else None
    return ExecResult(ok=bool(res.success) and err is None, result=result, stdout=cap.stdout, error=err)

def set_value(accessor, expr):
    "Assign `expr` (evaluated in user_ns) to the lvalue at `accessor`; return error text or None."
    ip = get_ipython()
    if ip is None: return 'no IPython kernel'
    try: _, path = _walk(ip.user_ns, tuple(accessor))
    except Exception as e: return f'bad accessor: {e}'
    try:
        exec(f'{path} = ({expr})', ip.user_ns)
        return None
    except Exception as e:
        return f'{type(e).__name__}: {e}'

In [ ]:
run(1+1)

AttributeError: 'int' object has no attribute 'isspace'

ExecResult(ok=False, result=None, stdout='', error="AttributeError: 'int' object has no attribute 'isspace'")

In [ ]:
def test_complete_captureshell_style():
    "execnb's CaptureShell.complete(c) returns a flat list of tail matches; complete() must use that shape."
    global get_ipython
    orig = get_ipython
    class _CapShell:   # mimics execnb CaptureShell / SmartCompleter: single-arg, tail-only matches
        def complete(self, c):
            tail = re.search(r'[\w.]*$', c).group(0).rpartition('.')[-1]
            return [m for m in ['Title', 'Titled', 'Div'] if m.startswith(tail)]
    get_ipython = lambda: _CapShell()
    try:
        r = complete('ft.Ti', 5)
        assert r['from']==3 and r['matches']==['Title','Titled']   # 'from' points at the 'Ti' tail after the dot
        assert complete('ft.', 3)['matches']==['Title','Titled','Div']   # bare dot -> all attrs
    finally:
        get_ipython = orig
test_complete_captureshell_style()

In [ ]:
import paar.runtime as RT, paar.bridge as BR
def test_ns_complete_headless():
    RT.get_ipython = lambda: None                       # force the no-kernel path
    BR.set_namespace({'demo_frame': 42, 'd': {'x': 1}})
    try:
        assert 'demo_frame' in RT.complete('dem', 3)['matches']    # namespace name
        assert 'keys' in RT.complete('d.', 2)['matches']           # attribute of a ns object
        assert 'zip' in RT.complete('zi', 2)['matches']            # builtin
    finally:
        BR.set_namespace(None)
test_ns_complete_headless()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()